## 🎯 Learning Objectives
* Understand the critical importance of cost and latency profiling for advanced LangGraph multi-agent systems.
* Learn how to instrument LangGraph agents and nodes for basic latency and resource consumption tracking.
* Interpret profiling outputs to identify performance bottlenecks and cost drivers.
* Explore modern tools and techniques for comprehensive profiling, including LangSmith and custom callback handlers.
* Discuss performance trade-offs and common use cases for profiling in production-grade agentic systems.


## Cost and Latency Profiling in LangGraph: Optimizing Advanced Agent Systems

Building sophisticated multi-agent systems with LangGraph is akin to designing a complex, interconnected city. Just as city planners must meticulously analyze traffic flow, resource consumption, and infrastructure efficiency to ensure a thriving metropolis, advanced AI engineers must profile their LangGraph applications for **cost** and **latency**. In 2026, as AI agents move from experimental prototypes to mission-critical production systems, understanding and optimizing these metrics is paramount.

### Why Profiling is Crucial for LangGraph

LangGraph's stateful execution, dynamic routing, and nested subgraphs introduce unique challenges for performance monitoring:

1.  **Dynamic Execution Paths**: Unlike linear programs, an agent's path through a LangGraph workflow can vary significantly based on user input, tool outputs, and LLM decisions. This makes predicting performance difficult without robust profiling.
2.  **External Dependencies**: Most agents rely on external LLM APIs, databases, search engines, or custom tools. Each of these introduces variable latency and direct costs.
3.  **Resource Consumption**: LLM calls consume tokens, which directly translate to monetary cost. Complex computations within nodes consume CPU/GPU and memory.
4.  **User Experience**: High latency directly impacts user satisfaction and the viability of real-time applications.
5.  **Operational Costs**: Unoptimized token usage or inefficient tool calls can lead to unexpectedly high cloud bills.

### The Analogy: City Traffic and Budget Management

Imagine your LangGraph application as a city. Each node is a district or a service (e.g., a police station, a library, a factory). The edges are roads connecting them. 

*   **Latency** is like traffic congestion. If one road (node execution) is always jammed, the entire city's flow slows down. Profiling helps you find these bottlenecks.
*   **Cost** is like the city's budget. Every service (LLM call, tool use) has an operational cost. Profiling helps you see where your budget is being spent and if it's being spent efficiently.

Without profiling, you're building a city without traffic cameras or financial audits – you'll only know there's a problem when the whole system grinds to a halt or the bills become astronomical.

### Key Metrics to Track

For LangGraph systems, we typically focus on:

*   **Node Latency**: Time taken for individual nodes (agents, tools, decision points) to execute.
*   **Graph Latency**: Total end-to-end time for a complete trace through the graph.
*   **LLM Token Usage**: Input and output tokens for each LLM call, directly correlating to cost.
*   **LLM API Cost**: Estimated or actual cost per LLM invocation.
*   **Tool Usage Cost**: Cost associated with external API calls or compute-intensive tools.
*   **Memory Footprint**: Especially relevant for long-running agents or large state objects.

### Modern Profiling Tools and Techniques (2026 Context)

1.  **LangSmith**: The gold standard for LangChain/LangGraph observability. It provides detailed traces, latency breakdowns, token usage, and cost estimates out-of-the-box. It's designed for multi-agent systems and offers powerful debugging and evaluation capabilities.
2.  **Custom Callback Handlers**: For fine-grained control or integration with existing systems, you can implement custom `BaseCallbackHandler` classes from `langchain_core.callbacks`. These allow you to hook into LLM, tool, and chain events to record metrics.
3.  **OpenTelemetry**: For distributed tracing across microservices or complex architectures, OpenTelemetry provides a vendor-agnostic standard for collecting telemetry data (traces, metrics, logs). LangChain/LangGraph can be integrated with OpenTelemetry for broader system observability.
4.  **Cloud Provider Monitoring**: Services like AWS CloudWatch, Azure Monitor, and Google Cloud Operations (formerly Stackdriver) can monitor the underlying compute resources (e.g., Lambda, Kubernetes pods) where your agents run, providing infrastructure-level metrics.

### Step-by-Step Profiling Approach

1.  **Instrument**: Add timing mechanisms (e.g., `time.perf_counter()`, decorators) or integrate with observability platforms (LangSmith, OpenTelemetry) into your nodes and LLM/tool calls.
2.  **Run Representative Workloads**: Execute your LangGraph application with typical inputs and scenarios to generate realistic traces.
3.  **Analyze Traces**: Use LangSmith's UI or custom dashboards to visualize execution paths, identify bottlenecks, and pinpoint high-cost operations.
4.  **Identify Bottlenecks**: Look for nodes with disproportionately high latency or LLM calls with excessive token usage.
5.  **Optimize**: Implement strategies like caching, prompt compression, model selection, parallelization, or more efficient tool implementations.
6.  **Iterate**: Re-profile after optimizations to verify improvements and detect new bottlenecks.


In [ ]:
import time
import os
from typing import Dict, Any, List
from functools import wraps

# LangGraph imports
from langgraph.graph import StateGraph, END, START
from langgraph.graph.message import BaseMessage, AnyMessage, add_messages

# LangChain imports for callbacks
from langchain_core.callbacks import BaseCallbackHandler
from langchain_core.messages import HumanMessage, AIMessage
from langchain_core.runnables import RunnableConfig

# --- Mock Components for Demonstration ---
# In a real scenario, these would be actual LLM clients and tool implementations.

class MockLLM:
    """A mock LLM that simulates response time and token usage."""
    def __init__(self, response_time: float = 0.1, cost_per_input_token: float = 0.00001, cost_per_output_token: float = 0.00003):
        self.response_time = response_time
        self.cost_per_input_token = cost_per_input_token
        self.cost_per_output_token = cost_per_output_token

    def invoke(self, prompt: str, config: Dict = None) -> str:
        time.sleep(self.response_time) # Simulate network latency and processing
        response = f"Mock LLM response to: {prompt[:50]}... (simulated)"
        
        # Simulate token usage and cost calculation
        input_tokens = len(prompt) // 4 # Rough estimate: 1 token ~ 4 chars
        output_tokens = len(response) // 4
        llm_cost = (input_tokens * self.cost_per_input_token) + (output_tokens * self.cost_per_output_token)

        # If a ProfilingCallbackHandler is present, update its metrics
        if config and "callbacks" in config:
            for callback in config["callbacks"]:
                if isinstance(callback, ProfilingCallbackHandler):
                    callback.on_llm_end(None, input_tokens=input_tokens, output_tokens=output_tokens, cost=llm_cost)
        return response

class MockTool:
    """A mock tool that simulates execution time and a fixed cost per use."""
    def __init__(self, response_time: float = 0.2, cost_per_use: float = 0.01):
        self.response_time = response_time
        self.cost_per_use = cost_per_use

    def invoke(self, query: str, config: Dict = None) -> str:
        time.sleep(self.response_time) # Simulate tool execution time
        
        # If a ProfilingCallbackHandler is present, update its metrics
        if config and "callbacks" in config:
            for callback in config["callbacks"]:
                if isinstance(callback, ProfilingCallbackHandler):
                    callback.on_tool_end(None, cost=self.cost_per_use) # Simulate tool cost
        return f"Mock Tool result for: {query[:50]}... (simulated)"

# --- Custom Profiling Callback Handler ---

class ProfilingCallbackHandler(BaseCallbackHandler):
    """A custom callback handler to collect profiling data during LangChain/LangGraph execution."""
    def __init__(self):
        self.llm_calls = []
        self.tool_calls = []
        self.total_llm_input_tokens = 0
        self.total_llm_output_tokens = 0
        self.total_llm_cost = 0.0
        self.total_tool_cost = 0.0
        self._current_llm_start_time = None
        self._current_tool_start_time = None

    def on_llm_start(self, serialized: Dict[str, Any], prompts: List[str], **kwargs: Any) -> None:
        """Called when an LLM run starts."""
        self._current_llm_start_time = time.perf_counter()

    def on_llm_end(self, response: Any, **kwargs: Any) -> None:
        """Called when an LLM run ends. Collects duration, tokens, and cost."""
        end_time = time.perf_counter()
        duration = end_time - (self._current_llm_start_time or end_time)
        input_tokens = kwargs.get("input_tokens", 0)
        output_tokens = kwargs.get("output_tokens", 0)
        cost = kwargs.get("cost", 0.0)
        
        self.llm_calls.append({"duration": duration, "input_tokens": input_tokens, "output_tokens": output_tokens, "cost": cost})
        self.total_llm_input_tokens += input_tokens
        self.total_llm_output_tokens += output_tokens
        self.total_llm_cost += cost
        self._current_llm_start_time = None

    def on_tool_start(self, serialized: Dict[str, Any], input_str: str, **kwargs: Any) -> Any:
        """Called when a tool run starts."""
        self._current_tool_start_time = time.perf_counter()

    def on_tool_end(self, output: Any, **kwargs: Any) -> Any:
        """Called when a tool run ends. Collects duration and cost."""
        end_time = time.perf_counter()
        duration = end_time - (self._current_tool_start_time or end_time)
        cost = kwargs.get("cost", 0.0)
        
        self.tool_calls.append({"duration": duration, "cost": cost})
        self.total_tool_cost += cost
        self._current_tool_start_time = None

    def get_metrics(self) -> Dict[str, Any]:
        """Returns a summary of collected profiling metrics."""
        return {
            "total_llm_input_tokens": self.total_llm_input_tokens,
            "total_llm_output_tokens": self.total_llm_output_tokens,
            "total_llm_cost": self.total_llm_cost,
            "total_tool_cost": self.total_tool_cost,
            "overall_estimated_cost": self.total_llm_cost + self.total_tool_cost,
            "llm_call_details": self.llm_calls,
            "tool_call_details": self.tool_calls,
        }

# --- LangGraph State and Nodes ---

# Define the state for our LangGraph agent
class AgentState(Dict):
    messages: List[BaseMessage]
    tool_calls: List[Dict]
    llm_response: str
    # We'll store node-specific durations here for simplicity, 
    # though LangSmith or a more complex callback would handle this automatically.
    node_durations: Dict[str, float]

# Initialize mock components
mock_llm = MockLLM(response_time=0.08) # Slightly faster LLM
mock_tool = MockTool(response_time=0.18) # Slightly slower tool

# Define the nodes of our LangGraph
def call_llm_node(state: AgentState, config: RunnableConfig) -> AgentState:
    """Node to call the LLM and update the state."""
    start_time = time.perf_counter()
    
    current_messages = state["messages"]
    prompt = current_messages[-1].content if current_messages else ""
    
    print(f"\n[LLM Node] Calling LLM with prompt: {prompt[:30]}...")
    llm_response = mock_llm.invoke(prompt, config=config) # Pass config for callbacks
    
    end_time = time.perf_counter()
    state["node_durations"]["llm_node"] = end_time - start_time

    return {"messages": add_messages(state["messages"], AIMessage(content=llm_response)), "llm_response": llm_response}

def call_tool_node(state: AgentState, config: RunnableConfig) -> AgentState:
    """Node to call a mock tool and update the state."""
    start_time = time.perf_counter()

    tool_query = state["llm_response"] # Assume LLM's response dictates tool query
    
    print(f"[Tool Node] Calling tool with query: {tool_query[:30]}...")
    tool_result = mock_tool.invoke(tool_query, config=config) # Pass config for callbacks
    
    end_time = time.perf_counter()
    state["node_durations"]["tool_node"] = end_time - start_time

    return {"messages": add_messages(state["messages"], HumanMessage(content=f"Tool result: {tool_result}")), "tool_calls": [{"name": "mock_tool", "result": tool_result}]}

def decide_action(state: AgentState) -> str:
    """Conditional edge function to decide the next action."""
    # Simple logic: if LLM response contains "tool" or "data", call tool, else finish
    if "tool" in state["llm_response"].lower() or "data" in state["llm_response"].lower():
        print("[Decision] Decided to call tool.")
        return "call_tool"
    else:
        print("[Decision] Decided to finish.")
        return "end"

# --- Build the LangGraph Workflow ---

workflow = StateGraph(AgentState)

workflow.add_node("llm_node", call_llm_node)
workflow.add_node("tool_node", call_tool_node)

workflow.set_entry_point("llm_node")

workflow.add_conditional_edges(
    "llm_node",
    decide_action,
    {"call_tool": "tool_node", "end": END}
)
workflow.add_edge("tool_node", END)

app = workflow.compile()

# --- Execution with Profiling ---
print("--- Running LangGraph with Custom Profiling ---")

# Initialize the custom profiler
profiler = ProfilingCallbackHandler()

# --- Optional: LangSmith Integration (2026 Best Practice) ---
# Uncomment and set your API key to see this in LangSmith UI
# os.environ["LANGCHAIN_TRACING_V2"] = "true"
# os.environ["LANGCHAIN_API_KEY"] = "YOUR_LANGSMITH_API_KEY_HERE"
# os.environ["LANGCHAIN_PROJECT"] = "LangGraph Profiling Demo ADV01-L12"
# print(f"LangSmith tracing enabled: {os.getenv('LANGCHAIN_TRACING_V2') == 'true'}")
# if os.getenv("LANGCHAIN_TRACING_V2") == "true":
#     print(f"View traces at: https://smith.langchain.com/o/{os.getenv('LANGCHAIN_PROJECT')}/traces")

initial_state = {
    "messages": [HumanMessage(content="Analyze the latest market data for AI agent frameworks and suggest a new feature for LangGraph.")],
    "tool_calls": [],
    "llm_response": "",
    "node_durations": {} # Initialize for node-specific timing
}

# Run the graph with the custom profiler as a callback
# LangGraph automatically passes the 'config' object (including callbacks) to nodes.
start_graph_time = time.perf_counter()
final_state = app.invoke(
    initial_state,
    config={
        "callbacks": [profiler], 
        "recursion_limit": 5 # Prevent infinite loops in complex graphs
    }
)
end_graph_time = time.perf_counter()

print("\n--- Profiling Results Summary ---")
metrics = profiler.get_metrics()

print(f"Total Graph Execution Time: {end_graph_time - start_graph_time:.4f} seconds")
print(f"Overall Estimated Cost: ${metrics['overall_estimated_cost']:.6f}")
print(f"  - Total LLM Estimated Cost: ${metrics['total_llm_cost']:.6f}")
print(f"  - Total Tool Estimated Cost: ${metrics['total_tool_cost']:.6f}")
print(f"Total LLM Input Tokens: {metrics['total_llm_input_tokens']}")
print(f"Total LLM Output Tokens: {metrics['total_llm_output_tokens']}")

print("\n--- Detailed Node Durations ---")
for node, duration in final_state["node_durations"].items():
    print(f"- {node}: {duration:.4f} seconds")

print("\n--- Detailed LLM Call Metrics ---")
for i, detail in enumerate(metrics['llm_call_details']):
    print(f"  Call {i+1}: Duration={detail['duration']:.4f}s, Input Tokens={detail['input_tokens']}, Output Tokens={detail['output_tokens']}, Cost=${detail['cost']:.6f}")

print("\n--- Detailed Tool Call Metrics ---")
for i, detail in enumerate(metrics['tool_call_details']):
    print(f"  Call {i+1}: Duration={detail['duration']:.4f}s, Cost=${detail['cost']:.6f}")

print("\n--- Final State Messages ---")
for msg in final_state["messages"]:
    print(msg)


### Interpreting the Profiling Output

The output from the code cell provides a granular view into the performance and cost characteristics of our simple LangGraph agent. Let's break down what each section tells us:

1.  **Total Graph Execution Time**: This is the overall wall-clock time taken for the entire LangGraph workflow to complete. It's a crucial high-level metric for user experience and SLA compliance.

2.  **Overall Estimated Cost**: This aggregates the simulated costs from all LLM and tool calls. In a real application, this would reflect your actual API expenses. A high cost here indicates a need to optimize token usage, choose cheaper models, or reduce unnecessary tool calls.

3.  **Total LLM Input/Output Tokens**: These metrics directly correlate to LLM costs. High input tokens might suggest verbose prompts or large context windows, while high output tokens could mean the LLM is generating overly detailed or redundant responses. Prompt engineering and response parsing can help here.

4.  **Detailed Node Durations**: This section shows how much time each individual node (`llm_node`, `tool_node` in our example) spent executing. Nodes with significantly higher durations are **bottlenecks**. In our example, `tool_node` might be slower than `llm_node`, indicating that the tool's external API call or internal processing is the primary contributor to latency.

5.  **Detailed LLM/Tool Call Metrics**: These provide per-call breakdowns. You can see the duration and cost for each specific interaction. This is invaluable for identifying specific LLM calls that are expensive or slow, or particular tool invocations that are problematic.

### Performance Trade-offs and Optimization Strategies

Profiling helps you make informed decisions about trade-offs:

*   **Cost vs. Latency**: Often, faster LLMs (e.g., larger, more capable models) come with a higher per-token cost. Conversely, cheaper models might be slower or less accurate. Your profiling data will guide you in selecting the right balance for your application's requirements.
*   **Complexity vs. Observability Overhead**: While tools like LangSmith are highly optimized, custom profiling adds some overhead. The benefit of deep insights usually outweighs this, but it's a factor to consider.
*   **Accuracy vs. Speed**: For some tasks, a slightly less accurate but much faster model might be acceptable if real-time response is critical.

**Common Optimization Strategies:**

*   **Caching**: Cache LLM responses or tool results for frequently asked questions or stable data. This drastically reduces latency and cost.
*   **Prompt Compression**: Use techniques like summarization or few-shot examples to reduce input token count without losing critical context.
*   **Model Selection**: Use smaller, cheaper, or fine-tuned models for specific tasks where a large general-purpose LLM is overkill.
*   **Parallelization**: If nodes are independent, explore running them in parallel using `async` operations or LangGraph's built-in parallel execution capabilities.
*   **Tool Optimization**: Improve the efficiency of your custom tools or choose faster external APIs.
*   **Batching**: For certain LLM calls or tool invocations, batching requests can improve throughput, though it might slightly increase individual request latency.
*   **Early Exit Conditions**: Design your graph to exit early when a sufficient answer is found, avoiding unnecessary computations.

### Typical Use Cases for Profiling in Production

*   **Continuous Monitoring**: Integrate profiling into your CI/CD pipeline and production monitoring dashboards to detect performance regressions or unexpected cost spikes immediately.
*   **SLA Compliance**: Ensure your agent system consistently meets its Service Level Agreements for response time.
*   **Cost Management**: Forecast and control your cloud and API expenses, preventing budget overruns.
*   **A/B Testing**: Compare different agent architectures, LLM providers, or prompt engineering strategies to empirically determine the most efficient approach.
*   **Root Cause Analysis**: Quickly pinpoint the source of performance issues or errors by examining detailed traces.

By systematically profiling your LangGraph applications, you transform them from opaque black boxes into transparent, optimizable systems, ready for the demands of 2026's advanced AI landscape.


### Resources for Further Learning

*   **LangSmith Documentation**: The official documentation for LangSmith, essential for advanced LangGraph observability.
    *   [https://docs.smith.langchain.com/](https://docs.smith.langchain.com/)
    *   [LangGraph Tracing with LangSmith](https://docs.smith.langchain.com/how_to_guides/langgraph_tracing)

*   **LangChain Callbacks Documentation**: Learn more about creating custom callback handlers for fine-grained control over tracing and metrics.
    *   [https://python.langchain.com/docs/modules/callbacks/](https://python.langchain.com/docs/modules/callbacks/)

*   **LangGraph Documentation**: Explore the core concepts of LangGraph, including state management and graph execution.
    *   [https://langchain-ai.github.io/langgraph/](https://langchain-ai.github.io/langgraph/)

*   **OpenTelemetry Python Documentation**: For distributed tracing and integrating with broader observability stacks.
    *   [https://opentelemetry.io/docs/instrumentation/python/](https://opentelemetry.io/docs/instrumentation/python/)

*   **Cloud Provider Monitoring**: Relevant documentation for monitoring your infrastructure where agents are deployed.
    *   **AWS CloudWatch**: [https://aws.amazon.com/cloudwatch/](https://aws.amazon.com/cloudwatch/)
    *   **Azure Monitor**: [https://azure.microsoft.com/en-us/products/monitor](https://azure.microsoft.com/en-us/products/monitor)
    *   **Google Cloud Operations (formerly Stackdriver)**: [https://cloud.google.com/products/operations](https://cloud.google.com/products/operations)
